# Bronze Quality Assessment — contratos_planos

Este notebook realiza o diagnóstico de qualidade da tabela `bronze.contratos_planos` antes da construção da camada Silver.

## Objetivos
- medir volume e integridade básica da tabela
- identificar duplicidades
- validar regras temporais
- verificar consistência de valores monetários
- mapear domínio de categorias

## Resultado esperado
Ao final deste notebook, deve estar claro:

- quais problemas existem na Bronze
- qual a magnitude de cada problema
- quais regras serão aplicadas na Silver


## Volume da tabela

Esta verificação mede o tamanho da tabela `contratos_planos` na camada Bronze.

Objetivo:
confirmar se a ingestão carregou o volume esperado de registros.

Essa métrica também serve como baseline para identificar problemas de carga em execuções futuras.

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM healthcare_dev.bronze.contratos_planos;

## Unicidade da chave

Verifica se o identificador `contrato_id` é único na tabela.

Objetivo:
identificar possíveis duplicidades geradas por erro de ingestão ou inconsistência na origem.

A chave `contrato_id` deve representar um contrato único.

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT contrato_id) AS contratos_unicos,
  COUNT(*) - COUNT(DISTINCT contrato_id) AS duplicados
FROM healthcare_dev.bronze.contratos_planos

## Duplicados detalhados

Caso existam duplicidades de `contrato_id`, esta consulta identifica quais contratos aparecem mais de uma vez.

Objetivo:
localizar os registros problemáticos para análise posterior na camada Silver.

In [0]:
%sql
SELECT
  contrato_id,
  COUNT(*) AS qtd
FROM healthcare_dev.bronze.contratos_planos
GROUP BY contrato_id
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

## Consistência temporal

Verifica se existem contratos cuja data de término é anterior à data de início.

Objetivo:
identificar inconsistências temporais que podem ter sido causadas por erro de sistema ou falha na origem dos dados.

Esses registros serão tratados ou descartados na camada Silver.

In [0]:
%sql
SELECT
  COUNT(*) AS contratos_datas_invalidas
FROM healthcare_dev.bronze.contratos_planos
WHERE data_fim_vigencia < data_inicio_vigencia;

## Valores monetários inválidos

Verifica se existem contratos com valor mensal negativo.

Objetivo:
identificar inconsistências financeiras que podem indicar erro de ingestão ou erro na origem dos dados.

Como os campos na Bronze são armazenados como STRING, utiliza TRY_CAST para converter para DOUBLE antes da validação.

In [0]:
%sql
SELECT
  COUNT(*) AS valores_negativos
FROM healthcare_dev.bronze.contratos_planos
WHERE TRY_CAST(valor_mensal AS DOUBLE) < 0;

## Domínio categórico

Analisa os valores existentes na coluna `tipo_plano`.

Objetivo:
identificar categorias existentes e possíveis variações de escrita.

Valores esperados:
- empresarial
- individual_familiar
- coletivo_adesao

Variações serão padronizadas na camada Silver.

In [0]:
%sql
SELECT
  tipo_plano,
  COUNT(*) AS qtd
FROM healthcare_dev.bronze.contratos_planos
GROUP BY tipo_plano
ORDER BY qtd DESC;

## Integridade de campos críticos

Verifica a presença de valores nulos em colunas consideradas essenciais para análise.

Campos avaliados:
- contrato_id
- beneficiario_id
- data_inicio_vigencia

A ausência desses valores compromete a confiabilidade da análise e exige tratamento na camada Silver.

In [0]:
%sql
SELECT
  SUM(CASE WHEN contrato_id IS NULL THEN 1 ELSE 0 END) AS contrato_id_nulo,
  SUM(CASE WHEN beneficiario_id IS NULL THEN 1 ELSE 0 END) AS beneficiario_id_nulo,
  SUM(CASE WHEN data_inicio_vigencia IS NULL THEN 1 ELSE 0 END) AS data_inicio_nula
FROM healthcare_dev.bronze.contratos_planos;

## Conclusão do diagnóstico

A tabela `bronze.contratos_planos` apresentou:

- 25.000 registros totais
- 0 duplicidades em contrato_id
- 250 contratos com inconsistência temporal
- 0 valores monetários negativos
- 3 categorias válidas de tipo de plano
- 0 nulos em chaves críticas

### Decisão para Silver
- contratos com datas inválidas serão enviados para quarentena
- tipo_plano será padronizado para domínio controlado
- valor_mensal será convertido para DOUBLE com TRY_CAST
